# 建立模型 Dataset（Kaggle，CPU 即可，只做一次）

把 `hunyuan3d2_kaggle_2xt4.ipynb` 要用的模型權重（SDXL、MV-Adapter、Hunyuan3D-2mv、Hunyuan3D-2 形狀與貼圖、去背模型）從 Hugging Face 下載，存成你自己的 **private Kaggle Dataset**。之後跑 2×T4 時用 Add Input 加入這個 Dataset，模型直接出現在 `/kaggle/input/…`，不用再下載、不花 GPU 時間。

**開始前**（右側 Session options）：Accelerator 選 **None**（CPU，不扣 GPU 額度）、**Internet on**；Secrets 勾選 `KAGGLE_API_TOKEN`（或 `KAGGLE_KEY`）和選用的 `HF_TOKEN`。然後 **Run All**，或 Save Version → Save & Run All 在背景跑。

模型更新（例如要加 turbo 版本）時，改區塊 1 的設定再跑一次，會上傳成同一個 Dataset 的新版本。

In [ ]:
# 1. Settings and download (about 30 GB without turbo models)
KAGGLE_USERNAME = ""  # optional; block 2 uses the user name of the API token
DATASET_SLUG = "imageto3d-models"  # becomes kaggle.com/datasets/<user>/imageto3d-models
INCLUDE_TURBO = False  # also download the turbo shape/paint models (about 15 GB more)
MODEL_DIR = "/tmp/imageto3d-models"
PIPELINE_REF = "claude/gallant-thompson-py38mf"

import os, shutil, subprocess, sys, time
from kaggle_secrets import UserSecretsClient

def secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub==0.36.0", "hf_xet", "-U", "kagglehub"], check=True)
if not os.path.exists("/tmp/colab-ImageTo3D"):
    subprocess.run(["git", "clone", "-q", "https://github.com/chousir/colab-ImageTo3D.git", "/tmp/colab-ImageTo3D"], check=True)
subprocess.run(f"git fetch -q origin {PIPELINE_REF} && git checkout -q -B run FETCH_HEAD", shell=True,
               cwd="/tmp/colab-ImageTo3D", check=True)
sys.path.insert(0, "/tmp/colab-ImageTo3D")
import imageto3d.models

print(f"Disk free: {shutil.disk_usage('/tmp').free / 1024**3:.0f} GB")
started = time.time()
imageto3d.models.download(MODEL_DIR, include_turbo=INCLUDE_TURBO, token=secret("HF_TOKEN"))
print(f"Download finished in {(time.time() - started) / 60:.0f} min")

In [ ]:
# 2. Upload as a private Kaggle dataset (new version if it already exists)
# Models are moved into one zip first (each file deleted once packed), so the disk never holds two copies.
import glob, os, shutil, subprocess, sys, time
subprocess.run(f"git fetch -q origin {PIPELINE_REF} && git checkout -q -B run FETCH_HEAD", shell=True,
               cwd="/tmp/colab-ImageTo3D", check=True)
import importlib, imageto3d.models
importlib.reload(imageto3d.models)
for stale in glob.glob("/tmp/tmp*/archive.zip"):  # left by an interrupted kagglehub upload
    shutil.rmtree(os.path.dirname(stale), ignore_errors=True)
token, key = secret("KAGGLE_API_TOKEN"), secret("KAGGLE_KEY")
if token:
    os.environ["KAGGLE_API_TOKEN"] = token
elif key and KAGGLE_USERNAME:
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = KAGGLE_USERNAME, key
else:
    raise RuntimeError("Add the secret KAGGLE_API_TOKEN (or KAGGLE_KEY plus KAGGLE_USERNAME in block 1): "
                       "kaggle.com/settings > API > Create New Token.")
import kagglehub
# The dataset owner must be the account of the token (the name in kaggle.com/<name>, not the display name).
OWNER = kagglehub.whoami()["username"]
if KAGGLE_USERNAME and KAGGLE_USERNAME != OWNER:
    print(f"KAGGLE_USERNAME {KAGGLE_USERNAME!r} is not the token's account; using {OWNER!r}")
print("Kaggle account:", OWNER)
UPLOAD_DIR = "/tmp/imageto3d-upload"
if os.path.isdir(MODEL_DIR):
    imageto3d.models.pack(MODEL_DIR, UPLOAD_DIR)
print(f"Disk free: {shutil.disk_usage('/tmp').free / 1024**3:.0f} GB")
started = time.time()
kagglehub.dataset_upload(f"{OWNER}/{DATASET_SLUG}", UPLOAD_DIR,
                         version_notes=f"imageto3d models, turbo={INCLUDE_TURBO}, {time.strftime('%Y-%m-%d')}")
print(f"Uploaded in {(time.time() - started) / 60:.0f} min: https://www.kaggle.com/datasets/{OWNER}/{DATASET_SLUG}")
print("Kaggle unpacks the zip; when the dataset page shows the files, check it is Private, then Add Input it.")